In [4]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold

random_state = 42
cv_spits = StratifiedKFold(n_splits=3,shuffle=True,random_state=random_state)
train_size = 0.67
np.random.seed(random_state)

# Explore Data

In [ ]:
target = 'target'
url = ''
df = pd.DataFrame(url)

df.shape

In [ ]:
df.head()

In [ ]:
df.describe()

In [ ]:
df.shape[0] - df.dropna().shape[0]

In [ ]:
df.boxplot(figsize=(15,12))
plt.show()

In [ ]:
sns.pairplot(df,hue=target)
plt.show()

# Preprocessing

In [ ]:
df.isna().sum()

df = df.dropna()

In [ ]:
target = ''
X = df.drop(target,axis=1)
y = df[target]

X.shape
y.shape

In [ ]:
y.value_counts().sort_index().plot(kind='bar',rot=0)

In [ ]:
from sklearn.preprocessing import LabelEncoder
column_to_encode = ''
le = LabelEncoder()
df[column_to_encode] = le.fit_transform(df[column_to_encode])

In [ ]:
from sklearn.preprocessing import OneHotEncoder

column_to_encode = ''
one = OneHotEncoder()
enc_data = one.fit_transform(df[column_to_encode])
new_columns = list(one.categories_[0])
enc_df = pd.DataFrame(enc_data.toarray(),columns=new_columns)
df = df.join(enc_df)
df = df.drop(column_to_encode, axis=1)
df.head()

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

column_to_encode=''
categories = []
oe = OrdinalEncoder(categories=categories,dtype=int)
df[column_to_encode] = oe.fit_transform(df[column_to_encode].values)

In [ ]:
from sklearn.preprocessing import MinMaxScaler
mms = MinMaxScaler()
X_transformed = pd.DataFrame(mms.fit_transform(X),columns=X.columns)

In [ ]:
from sklearn.preprocessing import PowerTransformer,StandardScaler
from sklearn.pipeline import make_pipeline

pipeline = make_pipeline(PowerTransformer(),StandardScaler())
X_transformed = pd.DataFrame(pipeline.fit_transform(X),columns=X.columns)

In [ ]:
from sklearn.decomposition import PCA

pca = PCA()
X_transformed = pca.fit_transform(X)
print(pca.explained_variance_ratio_)
min_variance = 0.9
variance_cumsum = np.cumsum(pca.explained_variance_ratio_.copy())
cuttoff_index = np.argmax(variance_cumsum>=min_variance)
X = X_transformed[:,:cuttoff_index+1]


### Training

In [ ]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(X,y,random_state=random_state,shuffle=True)

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier,RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import Perceptron
from sklearn.neighbors import KNeighborsClassifier

model_lbls = [
    'dt',
    'nb',
    'knn',
    'lp',
    'rf',
    'adb'

    # 'svm'
]


param_dt = [{'max_depth':[*range(1,20)], 'class_weight':[None,'balanced']}]
param_nb = [{'var_smoothing':[10**exp for exp in range(-3,-12,-1)]}]
param_knn = [{'n_neighbors':[*range(2,10)]}]
param_lp = [{'early_stopping':[True,False], 'class_weight':[None,'balanced']}]
param_adb = [{'n_estimators':[*range(10,51,10)],'learning_rate':[0.5,0.75,1,1.25,1.5]}]
param_rf = [{'n_estimators':[*range(10,31,5)],'max_depth':[*range(4,30,4)],'class_weight':[None,'balanced']}]

param_svm = [
    {'kernel':['rbf'],'gamma':[1e-3,1e-4],'C':[1,10,100]},
    {'kernel':['linear'],'C':[1,10,100]}
]


models = {
    'dt':{
        'name': 'Decision Tree',
        'estimator': DecisionTreeClassifier(random_state=random_state),
        'param': param_dt
    },
    'nb':{
        'name':'Naive Bayers',
        'estimator': GaussianNB(),
        'param': param_nb,
    },
    'knn': {
        'name': 'K nearest neighbors',
        'estimator': KNeighborsClassifier(),
        'param': param_knn
    },
    'lp':{
        'name': 'Linear perceptron',
        'estimator': Perceptron(random_state=random_state),
        'param': param_lp
    },
    'rf':{
        'name': 'RandomForestClassifier',
        'estimator': RandomForestClassifier(random_state=random_state),
        'param': param_rf
    },
    'adb':{
        'name': 'AdaBoostClassifier',
        'estimator': AdaBoostClassifier(random_state=random_state),
        'param': param_adb
    },

}

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report

scorings = ['accuracy','precision_macro','recall_macro','f1_macro']

clfs = []
results = pd.DataFrame([],columns=['scoring','model','best_param','accuracy','precision_macro','recall_macro','f1_macro'])

for score in scorings:
    for m in model_lbls:
        clf = GridSearchCV(
            estimator=models[m]['estimator'],
            param_grid=models[m]['param'],
            scoring=score,
            cv=cv_spits
        )
        clf.fit(X_train,y_train)
        clfs.append(clf)
        y_pred = clf.predict(X_test)
        cr = classification_report(y_test,y_pred)
        results.loc(len(results)) = [
            score,
            models[m]['name'],
            clf.best_params_,
            cr['accuracy'],
            cr['macro avg']['precision'],
            cr['macro avg']['recall'],
            cr['macro avg']['f1-score']
        ]

# Display Results

In [ ]:
for score in scorings:
    display(
        results[results.scorings==score]
        .sort_values(by='scorings',ascending=False)
        .drop('scorings',axis=1)
        .style.format(precision=3)
        .set_caption('ddosjsjdo')
    )

from sklearn.metrics import ConfusionMatrixDisplay

for score in scorings:
    best_row = results.loc[results.scoring==score,score].idxmax(axis=0)
    disp = ConfusionMatrixDisplay.from_estimator(estimator=clfs[best_row],X=X_test,y=y_test)